# B3: Crop Harvest Spatialization (GGCMI calendar)

Parse GGCMI Phase 3 crop-calendar data and extract the sowing–maturity window per grid cell.
- Source: GGCMI Phase 3 crop calendar NetCDF (Zenodo 5062513)
- Output: sowing/maturity day per crop per grid cell

In [ ]:
import yaml
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
from rasterio.enums import Resampling
import rioxarray
from geocube.api.core import make_geocube
import matplotlib.pyplot as plt

import sys
sys.path.append(str(Path('../src')))
from grid_utils import create_target_grid

with open(Path('../config.yaml')) as f:
    cfg = yaml.safe_load(f)

In [ ]:
for k, v in cfg['paths'].items():
    cfg["paths"][k] = Path(v)#.expanduser().resolve()

cfg["data_root"] = Path(cfg["data_root"])

# Inspect data structure

In [ ]:
ds = xr.open_dataset(cfg["paths"]["cropgrids"] / "CROPGRIDSv1.08_NC_maps/CROPGRIDSv1.08_wheat.nc")
(ds["harvarea"] / ds["croparea"].where(lambda x: x > 0)).plot.imshow(figsize=(4, 3))

In [ ]:
da_croparea = ds["croparea"]
# .rio.to_raster(cfg["paths"]["output"] / "wheat_area.tif", compress="lzw")

## Verify whether GGCMI croparea is the actual planted area (ha) per grid cell

In [ ]:
lat = da_croparea.lat.values
lon = da_croparea.lon.values

dlat = 0.05  # degrees
dlon = 0.05  # degrees
R = 6371000  # Earth radius, m

lat_rad = np.deg2rad(lat)
cell_area_m2 = (R**2 
                * np.deg2rad(dlon) 
                * np.abs(np.sin(lat_rad + np.deg2rad(dlat/2)) 
                       - np.sin(lat_rad - np.deg2rad(dlat/2))))
cell_area_ha = cell_area_m2 / 10000  # to hectares

# broadcast to the same shape as harvarea
cell_area_da = xr.DataArray(
    np.broadcast_to(cell_area_ha[:, None], da_croparea.shape),
    dims=da_croparea.dims,
    coords=da_croparea.coords
)

# key check: the ratio should be within [0,1]
ratio = da_croparea / cell_area_da

print("ratio stats:")
print(f"  max: {float(ratio.max()):.3f}")
print(f"  cells >1: {int((ratio > 1).sum())}")  # should be none

# latitude-gradient check: cell_area differences across latitude bands within China
print("\ncell area at different latitudes (ha):")
for lat_val in [20, 30, 40, 50]:
    idx = np.argmin(np.abs(lat - lat_val))
    print(f"  {lat_val}°N: {cell_area_ha[idx]:.0f} ha")

In [ ]:
cfg["paths"]["ggcmi"]


crop_name = "ri1_ir"
nc4_name = f"{crop_name}_ggcmi_crop_calendar_phase3_v1.01.nc4"

In [ ]:
ds_calendar = xr.open_dataset(cfg["paths"]["ggcmi"] / nc4_name, decode_timedelta=False)
"fraction_of_harvested_area" in ds_calendar.data_vars

In [ ]:
for _nc4 in cfg["paths"]["ggcmi"].glob("*_ggcmi_crop_calendar_phase3_v1.01.nc4"):
    ds_calendar = xr.open_dataset(_nc4, decode_timedelta=False)
    crop_name = _nc4.stem.split("_ggcmi_")[0]
    _fraction_of_harvested_area = "fraction_of_harvested_area" in ds_calendar.data_vars
    print(f"{crop_name}: {_fraction_of_harvested_area}")


In [ ]:
ds = xr.open_dataset(cfg["paths"]["cropgrids"] / "CROPGRIDSv1.08_NC_maps/CROPGRIDSv1.08_maize.nc")
(ds["harvarea"] / ds["croparea"].where(lambda x: x > 0)).plot.imshow()

In [ ]:
print(ds)

In [ ]:
# GGCMI structure
ds_ggcmi = xr.open_dataset(cfg["paths"]["ggcmi"] / "wwh_ir_ggcmi_crop_calendar_phase3_v1.01.nc4")
print(ds_ggcmi)

# monthly PSN structure
ds_psn = xr.open_dataarray(cfg["paths"]["gee_exports"] / "PSN_monthly_2011.tif")
print(ds_psn)

# Standard world-map DataArray

In [ ]:
gdf_world_merged = (
    gpd.read_parquet(cfg["paths"]["output"] / "world_clean.parquet")
        .query("NE_name != 'Antarctica'")
        .reset_index(drop=True)
        .assign(idx=lambda x: np.arange(1, x.shape[0]+1))    
)

In [ ]:
da_template = create_target_grid()
da_countries = make_geocube(gdf_world_merged, measurements=["idx"], like=da_template)["idx"].fillna(0)

# Convert FAO single-crop yield to NPP

In [ ]:
_years = list(range(2010, 2025))
FAO_m49_lst = gdf_world_merged['FAO_m49'].dropna().unique().tolist()
year_cols = [f"Y{y}" for y in _years]
meta_cols = ["Area Code", "Area Code (M49)", "Area", "Item Code", "code", "Item", "Element Code", "Element", "Unit"]
dic_year_rename = {f"Y{y}": y for y in _years}
dic_FAO2region = gdf_world_merged.set_index("FAO_name")["PNAS_region"].to_dict()

In [ ]:
df_crop_classification = pd.read_csv("../data/params/crop_classification.csv", index_col=0)


In [ ]:
df_harv_f = pd.read_excel(Path("../data/params/crop_coefficients.xlsx"), sheet_name="harvest factor")\
    .rename(columns={"harvest factor": "Harvest_cls"})\
    .melt(id_vars="Harvest_cls").rename(columns={"variable": "PNAS_region", "value": "hf"})
    
df_rec_r = pd.read_excel(Path("../data/params/crop_coefficients.xlsx"), sheet_name="recovery rate")\
        .rename(columns={"recovery rate": "Recover_cls"})\
        .melt(id_vars="Recover_cls").rename(columns={"variable": "PNAS_region", "value": "rr"})
        
dic_moisture = pd.read_csv(Path("../data/params/crop_moisture.csv")).set_index("Item")["moisture"].to_dict()

In [ ]:
exclude_item_codes = ['01921.02', '0143', '01491.02']

In [ ]:
df_crop_all = (
    pd.read_csv(cfg["paths"]["fao"] / "2024/production/Production_Crops_Livestock_E_All_Data_NOFLAG.csv", encoding_errors="ignore")
    .query("Element == 'Production'")
    .query("Unit != '1000 No'")
    .query("`Area Code (M49)` in @FAO_m49_lst")
    .rename(columns={"Item Code (CPC)": "code"})
    .assign(code=lambda x: x["code"].str.replace("'", ""))
    [meta_cols + year_cols]
    .dropna(subset=year_cols, how="all")
    .rename(columns=dic_year_rename)
    .query("code.str.startswith('01')") # 01 Products of agriculture, horticulture and market gardening
    # 011 Cereals
    .query("~code.str.startswith('012')") # 012 Vegetables
    .query("~code.str.startswith('013')") # 013 Fruits and nuts
    # 014 Oilseeds and oleaginous fruits
    # .query("~code.str.startswith('015')") # 015 Edible roots and tubers with high starch or inulin content
    .query("~code.str.startswith('016')") # 016 Stimulant, spice and aromatic crops
    # 017 Pulses (dried leguminous vegetables)
    # 018 Sugar crops
    # 019 Forage products; fibre crops; plants used in perfumery, pharmacy, or for insecticidal, fungicidal or similar purposes; beet, forage plant and flower seeds; natural rubber; living plants, cut flowers and flower buds; unmanufactured tobacco; other raw vegetable materials
    .query("~code.isin(@exclude_item_codes)")
    .reset_index(drop=True)
    .assign(PNAS_region=lambda x: x['Area'].map(dic_FAO2region))
    .merge(df_crop_classification, on="Item", how="left")
    .merge(df_harv_f, on=["Harvest_cls", "PNAS_region"], how="left")
    .merge(df_rec_r, on=["Recover_cls", "PNAS_region"], how="left")
    .assign(Moisture=lambda x: x['Item'].map(dic_moisture))
    .drop(columns=["Area Code", "Area Code (M49)", "Item Code", "Element Code", "Element", "Harvest_cls", "Recover_cls"])
)


In [ ]:
df_cropprod_DM_g = (
    df_crop_all
    .drop(columns=["Unit"])
    .pipe(lambda df: pd.concat([
        df.drop(columns=[*_years]),
        # fresh weight -> dry weight, t -> g
        df[_years].fillna(0).mul((1 - df["Moisture"] / 100) * 1e6, axis=0)
    ], axis=1))
)

df_residue_DM_g = (
    df_cropprod_DM_g.copy()
    .pipe(lambda df: pd.concat([
        df.drop(columns=_years),
        # yield -> residue
        df[_years].mul(df["hf"], axis=0)
    ], axis=1))
)

df_crop_harv_C = (
    df_cropprod_DM_g.copy()
    .pipe(lambda df: pd.concat([
        df.drop(columns=_years),
        # dry matter -> C
        (df[_years] + df_residue_DM_g[_years]) * 0.5
    ], axis=1))
    .drop(columns=["Moisture", "hf", "rr", "code"])
)

In [ ]:
df_FAO_CROPGRIDS_GGCMI = pd.read_csv("../data/params/FAO-CROPGRIDS-GGCMI.csv")\
    .assign(
        ggcmi=pd.col("ggcmi_phases").astype(str).str.split(","),
        fao_code=lambda x: x["fao_code"].astype(int)
    )


In [ ]:
year = 2015
fao_item = "Wheat"
cropgrid = df_FAO_CROPGRIDS_GGCMI.query("fao_item == @fao_item")["cropgrids_file"].values[0]
ggcmi_phases = df_FAO_CROPGRIDS_GGCMI.query("fao_item == @fao_item")["ggcmi"].values[0]
fao_code = df_FAO_CROPGRIDS_GGCMI.query("fao_item == @fao_item")["fao_code"].values[0]
df_FAO = (
    df_crop_harv_C
        .query("Item == @fao_item")
        .set_index("Area")
        .join(gdf_world_merged.set_index("FAO_name")["idx"])
        .drop(columns=["PNAS_region", "Item"])
        .sort_values("idx")
        .reset_index()
        [["Area", "idx", year]]
)
fao_annual = df_FAO.set_index("idx")[year].to_dict()


# Wheat worked example

## 0. Prepare test parameters

In [ ]:
# ============================================================
# 0. prepare test parameters
# ============================================================
year = 2015
fao_item = "Wheat"
row = df_FAO_CROPGRIDS_GGCMI.query("fao_item == @fao_item").iloc[0]
crop      = row["cropgrids_file"]       # "wheat"
phases    = row["ggcmi"]                # ["wwh", "swh"]
ggcmi_keys = [f"{p}_{ir}" for p in phases for ir in ("ir", "rf")]

# FAO annual carbon totals dict: {country_idx: gC}
df_FAO_test = (
    df_crop_harv_C
    .query("Item == @fao_item")
    .set_index("Area")
    .join(gdf_world_merged.set_index("FAO_name")["idx"])
    .reset_index()
    [["Area", "idx", year]]
)
fao_annual = df_FAO_test.set_index("idx")[year].to_dict()

print(f"crop: {crop},  GGCMI keys: {ggcmi_keys}")
print(f"n FAO countries with data: {sum(1 for v in fao_annual.values() if v > 0)}")
print(f"FAO global carbon total: {sum(v for v in fao_annual.values() if v > 0):.2e} gC")

## 1. Load PSN (monthly net photosynthesis)

In [ ]:
# ============================================================
# 1. Load PSN (monthly net photosynthesis)
# ============================================================
da_template = create_target_grid()
da_day_in_month = xr.DataArray([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], dims=["month"], coords={"month": np.arange(1, 13)})

raw_psn = xr.open_dataarray(
    cfg["paths"]["gee_exports"] / f"PSN_monthly_{year}.tif"
)
psn = (
    raw_psn.drop_attrs()
    .rio.write_crs("EPSG:4326")
    .assign_coords(band=np.arange(1, 13))
    .rename({"band": "month"})
    .clip(min=0)
    .rio.reproject_match(da_template, resampling=Resampling.nearest)
    * da_day_in_month
)

print(f"PSN shape: {psn.shape}")
print(f"PSN Jul global mean: {float(psn.sel(month=7).mean()):.2f} gC/m2")
print(f"PSN range: [{float(psn.min()):.1f}, {float(psn.max()):.1f}]")

psn.sel(month=7).plot.imshow(figsize=(8, 3), robust=True)
plt.title("PSN July")

## 2. Load CROPGRIDS harvest area


In [ ]:
# ============================================================
# 2. Load CROPGRIDS harvest area
# ============================================================
ds_cg = xr.open_dataset(
    cfg["paths"]["cropgrids"] / f"CROPGRIDSv1.08_NC_maps/CROPGRIDSv1.08_{crop}.nc",
    decode_times=False,
)
harvarea = (
    ds_cg["harvarea"]
    .where(ds_cg["harvarea"] >= 0, 0)
    .coarsen(lat=2, lon=2, boundary="trim").sum()
    .rio.write_crs("EPSG:4326")
    .rio.set_spatial_dims(x_dim="lon", y_dim="lat")
    .rio.reproject_match(da_template, resampling=Resampling.nearest)
    .fillna(0)
)

print(f"harvarea shape: {harvarea.shape}")
print(f"global total harvest area: {float(harvarea.sum()):.2e} ha")
print(f"non-zero pixels: {int((harvarea > 0).sum())}")

harvarea.where(harvarea > 0).plot.imshow(figsize=(4, 2), robust=True)
plt.title(f"CROPGRIDS {crop} harvarea (ha)")

## 3. Load GGCMI calendar & inspect planting/maturity DOY


In [ ]:
# ============================================================
# 3. Load GGCMI calendar & inspect planting/maturity DOY
# ============================================================
MONTH_END_DOY   = np.array([31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334, 365])
MONTH_START_DOY = np.array([ 1, 32, 60,  91, 121, 152, 182, 213, 244, 274, 305, 335])

ggcmi_data = {}
for key in ggcmi_keys:
    ds_g = xr.open_dataset(
        cfg["paths"]["ggcmi"] / f"{key}_ggcmi_crop_calendar_phase3_v1.01.nc4",
        decode_timedelta=False,
    )
    def _reproj(var):
        return (ds_g[var].rio.write_crs("EPSG:4326")
                .rio.set_spatial_dims(x_dim="lon", y_dim="lat")
                .rio.reproject_match(da_template, resampling=Resampling.bilinear).values)

    p = np.round(_reproj("planting_day")).astype(np.float32)
    m = np.round(_reproj("maturity_day")).astype(np.float32)
    frac = (
        _reproj("fraction_of_harvested_area").astype(np.float32)
        if "fraction_of_harvested_area" in ds_g.data_vars
        else np.ones_like(p)
    )
    frac = np.where(np.isfinite(frac), frac, 0.0)
    valid = np.isfinite(p) & np.isfinite(m)
    ggcmi_data[key] = {"planting": p, "maturity": m, "fraction": frac, "valid": valid}

    print(f"{key}: valid={valid.sum():,}px,  planting=[{np.nanmin(p):.0f},{np.nanmax(p):.0f}],  "
          f"maturity=[{np.nanmin(m):.0f},{np.nanmax(m):.0f}],  "
          f"fraction=[{np.nanmin(frac):.2f},{np.nanmax(frac):.2f}]")

In [ ]:
ax = plt.imshow(ggcmi_data["wwh_ir"]["planting"] < 50)
ax.axes.invert_yaxis()
plt.colorbar()

## 4. Verify _in_season_weight on a single key


In [ ]:
# ============================================================
# 4. Verify _in_season_weight on a single key
# ============================================================
def in_season_weight(planting, maturity):
    """Standalone version of _in_season_weight for step-by-step testing."""
    p = planting
    m = ((maturity - 1) % 365) + 1
    cross = (p > m)
    weights = np.zeros((12,) + p.shape, dtype=np.float32)
    for mo in range(12):
        s, e = MONTH_START_DOY[mo], MONTH_END_DOY[mo]
        ml = float(e - s + 1)
        # within-year
        ov_n = np.clip(np.minimum(e, m) - np.maximum(s, p) + 1, 0, None)
        # cross-year
        ov_c = (np.clip(np.minimum(e, 365) - np.maximum(s, p) + 1, 0, None)
              + np.clip(np.minimum(e, m) - np.maximum(s, 1) + 1, 0, None))
        weights[mo] = np.where(cross, ov_c, ov_n) / ml
    return weights

test_key = ggcmi_keys[0]   # "wwh_ir"
cal = ggcmi_data[test_key]
w = in_season_weight(cal["planting"], cal["maturity"])
print(f"in_season_weight shape: {w.shape}")

# Test point: North China Plain ~37N, 115E (winter wheat: sow ~Oct, harvest ~Jun)
iy = np.argmin(np.abs(da_template.y.values - 37))
ix = np.argmin(np.abs(da_template.x.values - 115))
p_pt = cal["planting"][iy, ix]
m_pt = cal["maturity"][iy, ix]
print(f"\nTest point (37N, 115E): planting DOY={p_pt:.0f}, maturity DOY={m_pt:.0f}")

    
plt.bar(np.arange(1, 13), w[:, iy, ix], )
print(f"  sum: {w[:, iy, ix].sum():.2f} (should ~ number of growing-season months)")

# 4-panel: Jan, Apr, Jun, Oct
fig, axes = plt.subplots(2, 2, figsize=(7, 3))
for i, mo_show in enumerate([1, 4, 6, 10]):
    ax = axes.flat[i]
    im = ax.imshow(
        np.where(cal["valid"], w[mo_show - 1], np.nan),
        extent=(da_template.x.min(), da_template.x.max(),
                da_template.y.min(), da_template.y.max()),
        vmin=0, vmax=1, cmap="YlGn",
    )
    ax.invert_yaxis()
    ax.set_title(f"{test_key} month={mo_show} weight")
plt.colorbar(im, ax=axes, shrink=0.6, label="weight")
plt.tight_layout()

In [ ]:
harv_vals = harvarea.values       # (ny, nx)
f = plt.imshow(harv_vals, extent=(da_template.x.min(), da_template.x.max(),
                              da_template.y.min(), da_template.y.max()))
f.axes.invert_yaxis()

## 5. Build harvest_weight (merge all keys)


In [ ]:
# ============================================================
# 5. Build harvest_weight (merge all keys)
# ============================================================
def doy_to_month0(doy):
    doy_w = ((np.asarray(doy) - 1) % 365) + 1
    return np.searchsorted(MONTH_END_DOY, doy_w - 1)

psn_vals = psn.values             # (12, ny, nx)
harv_vals = harvarea.values       # (ny, nx)
ny, nx = harv_vals.shape
hw = np.zeros((12, ny, nx), dtype=np.float64)

for key, cal in ggcmi_data.items():
    valid   = cal["valid"]
    in_seas = in_season_weight(cal["planting"], cal["maturity"])
    in_seas[:, ~valid] = 0.0

    cube = (psn_vals 
            * in_seas
            * harv_vals[np.newaxis, :, :]
            * cal["fraction"][np.newaxis, :, :])

    pw      = cube.sum(axis=0)
    harv_mo = doy_to_month0(cal["maturity"])

    for mo in range(12):
        hw[mo] += np.where((harv_mo == mo) & valid, pw, 0)

    print(f"{key}: nonzero pw={int((pw>0).sum()):,},  "
          f"max={pw.max():.1f},  harvest_month dist={[int(((harv_mo==m)&valid).sum()) for m in range(12)]}")

print(f"\nharvest_weight shape: {hw.shape}")
print(f"hw monthly global sums: {[f'{hw[m].sum():.2e}' for m in range(12)]}")

In [ ]:
f = plt.imshow(pw, extent=(da_template.x.min(), da_template.x.max(),
                              da_template.y.min(), da_template.y.max()))
f.axes.invert_yaxis()
plt.colorbar()

In [ ]:
f = plt.imshow(harv_mo, extent=(da_template.x.min(), da_template.x.max(),
                              da_template.y.min(), da_template.y.max()))
f.axes.invert_yaxis()
plt.colorbar()

## 6. Visual check: harvest_weight spatial distribution


In [ ]:
# ============================================================
# 6. Visual check: harvest_weight spatial distribution
# ============================================================
fig, axes = plt.subplots(3, 4, figsize=(16, 8))
vmax_val = np.nanpercentile(hw[hw > 0], 99) if (hw > 0).any() else 1
for mo in range(12):
    ax = axes.flat[mo]
    data = np.where(hw[mo] > 0, hw[mo], np.nan)
    im = ax.imshow(data,
              extent=(da_template.x.min(), da_template.x.max(),
                      da_template.y.min(), da_template.y.max()),
              cmap="YlOrRd", vmin=0, vmax=vmax_val)
    ax.invert_yaxis()
    ax.set_title(f"month {mo+1}", fontsize=10)
    ax.tick_params(labelsize=7)
plt.suptitle(f"harvest_weight by month (all ggcmi keys merged)", y=1.01)
plt.tight_layout()
plt.colorbar(im)

## 7. Vectorized allocation via bincount

In [ ]:
# ============================================================
# 7. Vectorized allocation via bincount
# ============================================================
mask = da_countries.fillna(0).values.astype(np.intp)   # ocean -> 0
n_ids = int(mask.max()+1)
flat_mask = mask.ravel()

# 7a. Zonal sum of hw by country
country_total = np.zeros(n_ids, dtype=np.float64)
for mo in range(12):
    country_total += np.bincount(flat_mask, weights=hw[mo].ravel(), minlength=n_ids)

print(f"countries with hw>0: {int((country_total > 0).sum())}")
# print(f"top-5 country_total: {np.sort(country_total)[-5:][::-1]}")

# 7b. FAO lookup array
fao_arr = np.zeros(n_ids, dtype=np.float64)
for code, total in fao_annual.items():
    code_int = int(code)
    if 0 <= code_int < n_ids:
        fao_arr[code_int] = total

# 7c. Core formula: result = hw * (FAO / total_hw)
safe_total = np.where(country_total > 0, country_total, 1.0)
ratio = (fao_arr / safe_total)[flat_mask].reshape(ny, nx)
result = (hw * ratio[np.newaxis, :, :]).astype(np.float32)

fao_sum = sum(v for v in fao_annual.values() if v > 0)
print(f"\nresult shape: {result.shape}")
print(f"result global total: {result.sum():.4e} gC")
print(f"FAO    global total: {fao_sum:.4e} gC")
print(f"gap: {abs(result.sum() - fao_sum) / fao_sum * 100:.2f}%  (countries w/o GGCMI/CROPGRIDS coverage)")

## CLASS

In [ ]:
"""
B4 - CropAllocator
Disaggregate FAO national annual crop carbon harvest to the 0.1deg grid x 12 months.

overall workflow:
    1. load PSN / CROPGRIDS / GGCMI, align to 0.1deg
    2. build harvest_weight[month, y, x]
       = sum_key ( PSN x in-season-day fraction x planted area x season share )
       placed into the month given by harvest_month
    3. vectorized allocation:
       result[mo, y, x] = FAO[country] × hw[mo,y,x] / Σhw[country]

    formula derivation (merging original Step3/4):
       orig Step3: share[mo,c] = sum_hw[mo,c] / sum_hw[c]      (monthly share)
       orig Step4: alloc = FAO[c] x share[mo] x hw_px / sum_hw[mo,c]
       merged:   = FAO[c] x (sum_hw[mo,c]/sum_hw[c]) x (hw_px/sum_hw[mo,c])
              = FAO[c] x hw_px / sum_hw[c]                  ← intermediate termcancels out

key assumptions:
    - carbon harvest occurs only in harvest_month (instantaneous-harvest assumption)
    - cells with higher in-season PSN receive more carbon (spatial allocation weight)
    - ir/rf share one CROPGRIDS harvarea (no irrigated/rainfed area split)

data sources:
    PSN       : GEE-exported MODIS MOD17A2HGF monthly net photosynthesis (gC/m2)
    CROPGRIDS : 0.05deg crop planted area (ha) -> coarsen to 0.1deg
    GGCMI     : 0.5deg sowing/harvest calendar (DOY) -> reproject to 0.1deg
    FAO       : national annual crop yield -> converted to carbon (gC)

usage:
    ca = CropAllocator(cfg, crop="wheat", ggcmi_phases=["wwh","swh"], year=2015)
    ca.load()
    result = ca.allocate(fao_annual, country_mask)   # → DataArray (12, ny, nx)
"""

class CropAllocator:
    """Disaggregate FAO national annual carbon harvest to the (month, y, x) grid."""

    # -- start/end DOY of each month (non-leap year) --
    MONTH_END_DOY   = np.array([31,  59,  90, 120, 151, 181,
                                212, 243, 273, 304, 334, 365])
    MONTH_START_DOY = np.array([ 1,  32,  60,  91, 121, 152,
                                182, 213, 244, 274, 305, 335])

    def __init__(self, cfg: dict, crop: str,
                 ggcmi_phases: list[str], year: int):
        """
        Parameters
        ----------
        cfg           : the loaded config.yaml dict, with paths / params
        crop          : CROPGRIDS crop name, e.g. "wheat", "rice", "maize"
        ggcmi_phases  : list of GGCMI calendar prefixes, e.g. ["wwh","swh"] or ["ri1","ri2"]
                        each phase auto-expands into _ir and _rf keys
        year          : data year, used to locate PSN_monthly_{year}.tif
        """
        self.crop         = crop
        self.year         = year
        self.ggcmi_phases = ggcmi_phases
        # expand to ["wwh_ir","wwh_rf","swh_ir","swh_rf"], etc.
        self.ggcmi_keys   = [
            f"{phase}_{irrf}"
            for phase in ggcmi_phases
            for irrf in ("ir", "rf")
        ]
        # target 0.1deg template grid (value = cell area m2, coords y/x)
        self.da_template = create_target_grid()
        self.target_y    = self.da_template.y.values
        self.target_x    = self.da_template.x.values

        # -- file paths --
        self.psn_path      = (Path(cfg["paths"]["gee_exports"])
                              / f"PSN_monthly_{year}.tif")
        self.cropgrids_dir = (Path(cfg["paths"]["cropgrids"])
                              / "CROPGRIDSv1.08_NC_maps")
        self.ggcmi_dir     = Path(cfg["paths"]["ggcmi"])

        # -- data slots to be filled by load() --
        self.psn      = None   # DataArray (12, ny, nx) -- monthly PSN
        self.harvarea = None   # DataArray (ny, nx)     -- planted area (ha)
        self.ggcmi    = {}     # {key: {planting, maturity, fraction, valid}}

    # ==================================================================
    # 1. load data & align to the 0.1deg target grid
    # ==================================================================

    def load(self):
        """Load PSN, CROPGRIDS, GGCMI in turn, all aligned to the 0.1deg template."""
        self._load_psn()
        self._load_cropgrids()
        self._load_ggcmi()

    # Days per month (non-leap), for converting daily-mean PSN to monthly total
    DAYS_IN_MONTH = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])

    def _load_psn(self):
        """
        load PSN_monthly_{year}.tif -> shape (12, ny, nx)
        - 12 bands for months 1-12
        - clip negatives to 0 (night/winter can give negative net photosynthesis)
        - nearest resampling to the 0.1deg template
        - PSN is a daily mean (gC/m2/day); multiply by days in month for the monthly total
        """
        days = xr.DataArray(
            self.DAYS_IN_MONTH, dims=["month"],
            coords={"month": np.arange(1, 13)},
        )
        raw = xr.open_dataarray(self.psn_path)
        self.psn = (
            raw
            .drop_attrs()
            .rio.write_crs("EPSG:4326")
            .assign_coords(band=np.arange(1, 13))
            .rename({"band": "month"})
            .clip(min=0)
            .rio.reproject_match(self.da_template, resampling=Resampling.nearest)
            * days   # daily mean -> monthly total
        )

    def _load_cropgrids(self):
        """
        load the harvarea variable of CROPGRIDSv1.08_{crop}.nc
        - native resolution 0.05deg, unit ha (hectares)
        - coarsen(2x2, sum)  -> 0.1deg (area is additive, so use sum)
        - replace negatives (no-data flag -1) with 0
        - nearest reprojection to the target template
        """
        ds = xr.open_dataset(
            self.cropgrids_dir / f"CROPGRIDSv1.08_{self.crop}.nc",
            decode_times=False
        )
        self.harvarea = (
            ds["harvarea"]
            .where(ds["harvarea"] >= 0, 0)       # -1 → 0
            .coarsen(lat=2, lon=2, boundary="trim")
            .sum()                                # 0.05deg -> 0.1deg, area sum
            .rio.write_crs("EPSG:4326")
            .rio.set_spatial_dims(x_dim="lon", y_dim="lat")
            .rio.reproject_match(self.da_template, resampling=Resampling.nearest)
            .fillna(0)
        )

    def _load_ggcmi(self):
        """
        Load the GGCMI Phase 3 crop calendar -> for each key:
          planting  (ny, nx) float32 — sowing-day DOY
          maturity  (ny, nx) float32 — harvest-day DOY
          fraction  (ny, nx) float32 — this season's share of total planted area
          valid     (ny, nx) bool    — cells with valid calendar data

        note:
        - original 0.5deg -> bilinear to 0.1deg, then round (dates must be integer DOY)
        - fraction exists only for multi-season crops (ri1/ri2); single-season defaults to 1.0
        """
        psn_shape = self.psn.shape[1:]   # (ny, nx)

        for key in self.ggcmi_keys:
            ds = xr.open_dataset(
                self.ggcmi_dir / f"{key}_ggcmi_crop_calendar_phase3_v1.01.nc4",
                decode_timedelta=False
            )

            def _reproject(var):
                """Bilinearly interpolate a GGCMI variable from 0.5deg to the 0.1deg target grid."""
                return (
                    ds[var]
                    .rio.write_crs("EPSG:4326")
                    .rio.set_spatial_dims(x_dim="lon", y_dim="lat")
                    .rio.reproject_match(self.da_template, resampling=Resampling.bilinear)
                    .values
                )

            planting = np.round(_reproject("planting_day")).astype(np.float32)
            maturity = np.round(_reproject("maturity_day")).astype(np.float32)

            # fraction: multi-season crops (ri1/ri2) have this, the season's share of total area
            # single-season crops lack this; fraction set to 1.0 (all harvarea to that season)
            if "fraction_of_harvested_area" in ds.data_vars:
                fraction = _reproject("fraction_of_harvested_area").astype(np.float32)
                fraction = np.where(np.isfinite(fraction), fraction, 0.0)
            else:
                fraction = np.ones(psn_shape, dtype=np.float32)

            self.ggcmi[key] = {
                "planting": planting,
                "maturity": maturity,
                "fraction": fraction,
                "valid":    np.isfinite(planting) & np.isfinite(maturity),
            }

    # ==================================================================
    # 2. build harvest_weight (merge all ggcmi keys)
    # ==================================================================

    def _in_season_weight(self, planting: np.ndarray,
                           maturity: np.ndarray) -> np.ndarray:
        """
        compute the **actual day-overlap fraction** between each month and the growing season.

        principle:
            weight = (days of the month within the growing season) / (total days in the month)
            e.g. sowing DOY=300, harvest DOY=60 (cross-year)
                10mo (274-304): overlap=5days -> 5/31~0.16
                11mo (305-334): overlap=30days -> 30/30=1.0
                 3mo (60-90):   overlap=1days -> 1/31~0.03

        cross-year handling:
            within-year (p <= m): growing season = [p, m]
            cross-year   (p > m): growing season = [p, 365] U [1, m]
            'cross-year' means sown in autumn and harvested the next spring (e.g. winter wheat)

        Parameters
        ----------
        planting : (ny, nx)  sowing-day DOY 1-365
        maturity : (ny, nx)  harvest-day DOY (>365 allowed, normalized by %365)

        Returns
        -------
        (12, ny, nx) float32  per-month growing-season overlap fraction, range [0, 1]
        """
        p     = planting
        m     = ((maturity - 1) % 365) + 1   # normalize to 1-365
        cross = (p > m)                       # True = cross-year growing season
        weights = np.zeros((12,) + p.shape, dtype=np.float32)

        for mo in range(12):
            s = self.MONTH_START_DOY[mo]      # first-day DOY of the month
            e = self.MONTH_END_DOY[mo]        # last-day DOY of the month
            month_len = float(e - s + 1)      # days in the month

            # -- within-year: [p, m] & [s, e] --
            overlap_normal = np.clip(
                np.minimum(e, m) - np.maximum(s, p) + 1, 0, None
            )

            # -- cross-year: ([p,365] & [s,e]) + ([1,m] & [s,e]) --
            # the two segments never overlap within one month (a month cannot contain both year start and end)
            overlap_p1 = np.clip(
                np.minimum(e, 365) - np.maximum(s, p) + 1, 0, None
            )
            overlap_p2 = np.clip(
                np.minimum(e, m) - np.maximum(s, 1) + 1, 0, None
            )
            overlap_cross = overlap_p1 + overlap_p2

            overlap = np.where(cross, overlap_cross, overlap_normal)
            weights[mo] = overlap / month_len

        return weights   # (12, ny, nx)

    def _doy_to_month0(self, doy: np.ndarray) -> np.ndarray:
        """
        convert DOY to a 0-based month index (0=Jan, 11=Dec).
        supports DOY>365 (normalized internally by %365 first).
        """
        doy_w = ((np.asarray(doy) - 1) % 365) + 1
        return np.searchsorted(self.MONTH_END_DOY, doy_w - 1)

    def build_harvest_weight(self) -> np.ndarray:
        """
        merge all ggcmi keys to build harvest_weight[12, ny, nx].

        for each key (e.g. wwh_ir):
            1. cube = PSN x in-season-day fraction x harvarea x fraction
            2. pixel_weight = cube.sum(month axis)  -> the key's annual production potential
            3. place pixel_weight into the month slot of harvest_month

        finally sum the results of all keys:
            harvest_weight[mo, y, x] = sum_key (pixel_weight_key where harvest_month==mo)

        meaning: total production-potential weight of cell (y,x) harvested in month mo.
        one cell may have weight in several months (e.g. winter wheat wwh and spring wheat swh harvest in different months).

        Returns
        -------
        (12, ny, nx) float64
        """
        psn      = self.psn.values        # (12, ny, nx)
        harvarea = self.harvarea.values   # (ny, nx)
        ny, nx   = harvarea.shape
        hw       = np.zeros((12, ny, nx), dtype=np.float64)

        for key, cal in self.ggcmi.items():
            valid   = cal["valid"]
            in_seas = self._in_season_weight(cal["planting"], cal["maturity"])
            in_seas[:, ~valid] = 0.0

            cube = (psn
                    * in_seas
                    * harvarea[np.newaxis, :, :]
                    * cal["fraction"][np.newaxis, :, :])

            pw = cube.sum(axis=0)                            # (ny, nx)
            harv_mo = self._doy_to_month0(cal["maturity"])   # (ny, nx) 0-based

            # place by harvest_month: put pw into the corresponding month slot
            for mo in range(12):
                hw[mo] += np.where((harv_mo == mo) & valid, pw, 0)

        return hw

    # ==================================================================
    # 3. vectorized allocation (bincount zonal stats, no per-country loop)
    # ==================================================================

    def allocate(self, fao_annual: dict,
                 country_mask: xr.DataArray) -> xr.DataArray:
        """
        allocate the FAO national annual carbon total to (month, y, x).

        core formula (simplified from the original Step3/4):
            result[mo, y, x] = FAO[c] × hw[mo, y, x] / Σhw[c]

            where c = country_mask[y, x]
            hw     = build_harvest_weight()  output
            sumhw[c] = sum of hw over all cells and months of the country

        simplification proof:
            orig Step3: share[mo] = sumhw[mo,c] / sumhw[c]
            orig Step4: px_result = FAO x share[mo] x hw_px / sumhw[mo,c]
                    = FAO × (Σhw[mo,c]/Σhw[c]) × (hw_px/Σhw[mo,c])
                    = FAO x hw_px / sumhw[c]    ← intermediate term sumhw[mo,c] cancels out

        Parameters
        ----------
        fao_annual   : {country_idx: annual carbon total gC}
        country_mask : (ny, nx) DataArray, values are country idx

        Returns
        -------
        DataArray (12, ny, nx)  per-cell per-month carbon harvest (gC)
        """
        hw   = self.build_harvest_weight()              # (12, ny, nx)
        mask = country_mask.fillna(0).values.astype(np.intp)  # NaN->0, (ny, nx) integer country id
        ny, nx = mask.shape
        n_ids  = int(mask.max()) + 1
        flat_mask = mask.ravel()

        # -- 1. zonal stats: sum hw by country -> country_total[country_id] --
        #    use np.bincount instead of per-country loop: O(n_pixels) not O(n_countries x n_pixels)
        country_total = np.zeros(n_ids, dtype=np.float64)
        for mo in range(12):
            country_total += np.bincount(
                flat_mask,
                weights=hw[mo].ravel(),
                minlength=n_ids
            )
        # country_total[c] = total harvest_weight of country c

        # -- 2. build FAO lookup array --
        fao_arr = np.zeros(n_ids, dtype=np.float64)
        for code, total in fao_annual.items():
            code_int = int(code)
            if 0 <= code_int < n_ids:
                fao_arr[code_int] = total

        # -- 3. vectorized: result = hw x (FAO / sum(hw)) per pixel --
        #    safe division: ratio=0 when sum(hw)=0 (no cultivation data; hw is also 0, so result is 0)
        safe_total = np.where(country_total > 0, country_total, 1.0)
        ratio = (fao_arr / safe_total)[flat_mask].reshape(ny, nx)  # (ny, nx)
        result = hw * ratio[np.newaxis, :, :]                      # (12, ny, nx)

        return xr.DataArray(
            result.astype(np.float32),
            dims=["month", "y", "x"],
            coords={"month": np.arange(1, 13),
                    "y": self.target_y,
                    "x": self.target_x},
            name=f"{self.crop}_HANPPharv",
        )

## 8. Cross-validation: functional vs class version

In [ ]:
# ============================================================
# 8. Diagnose: find EXACT divergence point between functional vs class
# ============================================================
ca_test = CropAllocator(cfg, crop=crop, ggcmi_phases=phases, year=year)
ca_test.load()

# --- 8a. Compare loaded data ---
print('=== Data loading ===')
print(f'PSN allclose:      {np.allclose(psn.values, ca_test.psn.values, equal_nan=True)}')
print(f'harvarea allclose: {np.allclose(harvarea.values, ca_test.harvarea.values, equal_nan=True)}')

for key in ggcmi_keys:
    f_cal = ggcmi_data[key]
    c_cal = ca_test.ggcmi[key]
    p_ok = np.allclose(f_cal['planting'], c_cal['planting'], equal_nan=True)
    m_ok = np.allclose(f_cal['maturity'], c_cal['maturity'], equal_nan=True)
    fr_ok = np.allclose(f_cal['fraction'], c_cal['fraction'], equal_nan=True)
    v_ok = np.array_equal(f_cal['valid'], c_cal['valid'])
    print(f'  {key}: plant={p_ok}, matur={m_ok}, frac={fr_ok}, valid={v_ok}')

# --- 8b. Compare build_harvest_weight ---
print('\n=== build_harvest_weight ===')
hw_class = ca_test.build_harvest_weight()
print(f'hw allclose: {np.allclose(hw, hw_class)}')
print(f'hw max abs diff: {np.abs(hw - hw_class).max():.6e}')
print(f'hw func sum: {hw.sum():.6e},  hw class sum: {hw_class.sum():.6e}')

# --- 8c. Compare allocate ---
print('\n=== allocate ===')
da_class = ca_test.allocate(fao_annual, da_countries)
print(f'result allclose: {np.allclose(result, da_class.values)}')
print(f'result max abs diff: {np.abs(result - da_class.values).max():.6e}')
print(f'func sum: {result.sum():.4e},  class sum: {da_class.values.sum():.4e}')

# --- 8d. Compare mask handling ---
print('\n=== mask ===')
func_mask = da_countries.fillna(0).values.astype(np.intp)
print(f'func mask min: {func_mask.min()},  max: {func_mask.max()}')
print(f'func mask has negative: {(func_mask < 0).any()}')
print(f'NaN pixels in da_countries: {int(np.isnan(da_countries.values).sum())}')

## 9. Conservation check: per-country FAO input vs grid output


# Object-oriented implementation

In [ ]:
df_FAO_CROPGRIDS_GGCMI = pd.read_csv("../data/params/FAO-CROPGRIDS-GGCMI.csv")\
    .assign(
        ggcmi=pd.col("ggcmi_phases").astype(str).str.split(","),
        fao_code=lambda x: x["fao_code"].astype(int)
    )


## Single staple-crop test

In [ ]:
year = 2015
fao_item = "Wheat"
fao_item = "Rice"
cropgrid = df_FAO_CROPGRIDS_GGCMI.query("fao_item == @fao_item")["cropgrids_file"].values[0]
ggcmi_phases = df_FAO_CROPGRIDS_GGCMI.query("fao_item == @fao_item")["ggcmi"].values[0]
fao_code = df_FAO_CROPGRIDS_GGCMI.query("fao_item == @fao_item")["fao_code"].values[0]
df_FAO = (
    df_crop_harv_C
        .query("Item == @fao_item")
        .set_index("Area")
        .join(gdf_world_merged.set_index("FAO_name")["idx"])
        .drop(columns=["PNAS_region", "Item"])
        .sort_values("idx")
        .reset_index()
        [["Area", "idx", year]]
)
fao_annual = df_FAO.set_index("idx")[year].to_dict()


In [ ]:
ca1 = CropAllocator(cfg, crop=cropgrid, ggcmi_phases=ggcmi_phases, year=2015)
ca1.load()
da_result = ca1.allocate(fao_annual, da_countries) / da_template # to gC/m2


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 8))
da_result_fl = da_result.values.flatten()
vmax_val = np.nanpercentile(da_result_fl[da_result_fl > 0], 99) if (da_result_fl > 0).any() else 1
for mo in range(12):
    ax = axes.flat[mo]
    data = np.where(da_result[mo] > 0, da_result[mo], np.nan)
    im = ax.imshow(data,
              extent=(da_template.x.min(), da_template.x.max(),
                      da_template.y.min(), da_template.y.max()),
              cmap="YlOrRd", vmin=0, vmax=vmax_val)
    ax.invert_yaxis()
    ax.set_title(f"month {mo+1}", fontsize=10)
    ax.tick_params(labelsize=7)
plt.tight_layout()
plt.colorbar(im)

## Oil palm

In [ ]:
class OilPalmAllocator(CropAllocator):
    """
    Oil-palm-specific allocator: uniform year-round harvest, no GGCMI calendar needed.

    temporal weight: 1/12 per month; spatial weight uses annual cumulative PSN x harvarea
    (perennials harvest year-round, but higher-productivity cells still receive more carbon)

    formula:
        hw[mo, y, x] = (1/12) × PSN_annual[y,x] × harvarea[y,x]

    differences from the main class:
        - _load_ggcmi: skip (no calendar data)
        - build_harvest_weight: distribute the PSN annual total evenly over 12 months
    """

    def __init__(self, cfg: dict, year: int):
        super().__init__(
            cfg,
            crop="oilpalm",
            ggcmi_phases=[],   # placeholder, not used
            year=year,
        )

    def _load_ggcmi(self):
        """Oil palm has no GGCMI calendar; skip."""
        self.ggcmi = {}

    def build_harvest_weight(self) -> np.ndarray:
        """
        uniform over the year: hw[mo] = PSN_annual x harvarea / 12

        Returns
        -------
        (12, ny, nx) float64
        """
        psn_annual = self.psn.values.sum(axis=0)          # (ny, nx) annual cumulative PSN
        base       = psn_annual * self.harvarea.values     # (ny, nx) spatial weight
        hw         = np.broadcast_to(
                         base[np.newaxis, :, :] / 12.0,
                         (12,) + base.shape
                     ).copy().astype(np.float64)
        return hw

In [ ]:
year = 2015
fao_item = "Oil palm fruit"
cropgrid = df_FAO_CROPGRIDS_GGCMI.query("fao_item == @fao_item")["cropgrids_file"].values[0]
fao_code = df_FAO_CROPGRIDS_GGCMI.query("fao_item == @fao_item")["fao_code"].values[0]
df_FAO = (
    df_crop_harv_C
        .query("Item == @fao_item")
        .set_index("Area")
        .join(gdf_world_merged.set_index("FAO_name")["idx"])
        .drop(columns=["PNAS_region", "Item"])
        .sort_values("idx")
        .reset_index()
        [["Area", "idx", year]]
)
fao_annual = df_FAO.set_index("idx")[year].to_dict()


In [ ]:
oa = OilPalmAllocator(cfg, year=year)
oa.load()
da_result = oa.allocate(fao_annual, da_countries) / da_template

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 8))
da_result_fl = da_result.values.flatten()
vmax_val = np.nanpercentile(da_result_fl[da_result_fl > 0], 99) if (da_result_fl > 0).any() else 1
for mo in range(12):
    ax = axes.flat[mo]
    data = np.where(da_result[mo] > 0, da_result[mo], np.nan)
    im = ax.imshow(data,
              extent=(da_template.x.min(), da_template.x.max(),
                      da_template.y.min(), da_template.y.max()),
              cmap="YlOrRd", vmin=0, vmax=vmax_val)
    ax.invert_yaxis()
    ax.set_title(f"month {mo+1}", fontsize=10)
    ax.tick_params(labelsize=7)
plt.tight_layout()
plt.colorbar(im)

In [ ]:
encoding = {
    var: {
        "dtype": "float32",
        "zlib": True,
        "complevel": 4,      # 1-9; 4 balances speed vs. size
        "chunksizes": (1, 180, 360),  # (time/month, y, x), optimized for monthly reads
    }
    for var in ds.data_vars
}


# Main pipeline

In [ ]:
year = 2015
def cal_harv_1year(year):
    data_vars = {}
    FAO_items = ['Wheat', 'Maize (corn)', 'Rice', 'Sorghum', 'Barley', 'Rye', 'Oats', 'Millet', 'Soya beans', 'Groundnuts, excluding shelled', 'Rape or colza seed', 'Sunflower seed', 'Sugar cane', 'Sugar beet', 'Seed cotton, unginned', 'Potatoes', 'Cassava, fresh', 'Sweet potatoes', 'Beans, dry', 'Peas, dry', 'Oil palm fruit']
    
    for fao_item in FAO_items:
        print(f"\n=== Processing {fao_item} ===")
        cropgrid    = df_FAO_CROPGRIDS_GGCMI.query("fao_item == @fao_item")["cropgrids_file"].values[0]
        ggcmi_phases = df_FAO_CROPGRIDS_GGCMI.query("fao_item == @fao_item")["ggcmi"].values[0]

        df_FAO = (
            df_crop_harv_C
                .query("Item == @fao_item")
                .set_index("Area")
                .join(gdf_world_merged.set_index("FAO_name")["idx"])
                .drop(columns=["PNAS_region", "Item"])
                .sort_values("idx")
                .reset_index()
                [["Area", "idx", year]]
        )
        fao_annual = df_FAO.set_index("idx")[year].to_dict()

        if fao_item == "Oil palm fruit":
            ca = OilPalmAllocator(cfg, year=year)
        else:
            ca = CropAllocator(cfg, crop=cropgrid, ggcmi_phases=ggcmi_phases, year=year)

        ca.load()
        data_vars[cropgrid] = ca.allocate(fao_annual, da_countries) / da_template

    ds_HANPPharv = xr.Dataset(data_vars)

    # save
    encoding = {
        var: {"dtype": "float32", "zlib": True, "complevel": 4, "chunksizes": (1, 180, 360)}
        for var in ds_HANPPharv.data_vars
    }
    ds_HANPPharv.to_netcdf(
        Path(cfg["paths"]["output"]) / f"HANPPharv_crops_{year}.nc",
        encoding=encoding
    )

In [ ]:
for year in range(2010, 2025):
    _nc_path = Path(cfg["paths"]["output"]) / f"HANPPharv_crops_{year}.nc"
    if _nc_path.exists():
        continue
    print(f"\n=== Calculating harvest for year {year} ===")
    cal_harv_1year(year)